# Data Warehouse Modeling (Star Schema)

## 1. Import Libraries & Connect To Clean Database

In [19]:
import pandas as pd
from sqlalchemy import create_engine

In [20]:

server = r".\SQLEXPRESS"
clean_database = "cleaning_railway"

clean_connection_string = (
   f"mssql+pyodbc://{server}/{clean_database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

clean_engine = create_engine(clean_connection_string)
df = pd.read_sql("SELECT * FROM clean_railway", clean_engine)

print("Clean data loaded successfully for Data Modeling!")

Clean data loaded successfully for Data Modeling!


## 2. Create Dimension Tables

## 1. Station Dimension

In [21]:
stations = pd.concat([df['Departure Station'], df['Arrival Destination']]).drop_duplicates().reset_index(drop=True)
dim_station = pd.DataFrame({'Station_Name': stations})
dim_station['Station_Key'] = dim_station.index + 1

dim_station.head()

,Station_Name,Station_Key
0,London Paddington,1
1,London Kings Cross,2
2,Liverpool Lime Street,3
3,London Euston,4
4,York,5


## 2. Ticket Dimension

In [22]:
dim_ticket = df[['Ticket Class', 'Ticket Type', 'Railcard']].drop_duplicates().reset_index(drop=True)
dim_ticket['Ticket_Key'] = dim_ticket.index + 1

dim_ticket.head()

,Ticket Class,Ticket Type,Railcard,Ticket_Key
0,Standard,Advance,Adult,1
1,Standard,Advance,No Railcard,2
2,Standard,Advance,Disabled,3
3,Standard,Advance,Senior,4
4,First Class,Advance,Senior,5


## 3. Payment Dimension

In [23]:
dim_payment = df[['Payment Method', 'Purchase Type']].drop_duplicates().reset_index(drop=True)
dim_payment['Payment_Key'] = dim_payment.index + 1

dim_payment.head()

,Payment Method,Purchase Type,Payment_Key
0,Contactless,Online,1
1,Credit Card,Station,2
2,Credit Card,Online,3
3,Contactless,Station,4
4,Debit Card,Station,5


## 4. Status Dimension

In [24]:
dim_status = df[['Journey Status', 'Reason for Delay', 'Refund Request']].drop_duplicates().reset_index(drop=True)
dim_status['Status_Key'] = dim_status.index + 1

dim_status.head()

,Journey Status,Reason for Delay,Refund Request,Status_Key
0,On Time,No Delay,No,1
1,Delayed,Signal Failure,No,2
2,Delayed,Technical Issue,Yes,3
3,Delayed,Signal Failure,Yes,4
4,Cancelled,Technical Issue,No,5


## 5. Date Dimension

In [25]:
dim_date = pd.DataFrame()
dim_date['Full_Date'] = pd.concat([df['Date of Purchase'], df['Date of Journey']]).drop_duplicates().reset_index(drop=True)
dim_date['Full_Date'] = pd.to_datetime(dim_date['Full_Date'], errors='coerce')
dim_date['Date_Key'] = dim_date.index + 1
dim_date['Day'] = dim_date['Full_Date'].dt.day
dim_date['Month'] = dim_date['Full_Date'].dt.month
dim_date['Year'] = dim_date['Full_Date'].dt.year
dim_date['Month_Name'] = dim_date['Full_Date'].dt.month_name()

dim_date.head()

,Full_Date,Date_Key,Day,Month,Year,Month_Name
0,2023-12-08,1,8,12,2023,December
1,2023-12-16,2,16,12,2023,December
2,2023-12-19,3,19,12,2023,December
3,2023-12-20,4,20,12,2023,December
4,2023-12-27,5,27,12,2023,December


## 6. Time Dimension

In [26]:
time_cols = ['Time of Purchase', 'Departure Time', 'Arrival Time', 'Actual Arrival Time']

all_times = pd.concat([df[col].dropna() for col in time_cols]).drop_duplicates().reset_index(drop=True)

dim_time = pd.DataFrame({'Time': all_times})
dim_time['Time_Key'] = dim_time.index + 1
dim_time['Hour'] = pd.to_datetime(dim_time['Time'], format='%H:%M:%S', errors='coerce').dt.hour
dim_time['Minute'] = pd.to_datetime(dim_time['Time'], format='%H:%M:%S', errors='coerce').dt.minute
dim_time['Seconds'] = pd.to_datetime(dim_time['Time'], format='%H:%M:%S', errors='coerce').dt.second

dim_time.head()

,Time,Time_Key,Hour,Minute,Seconds
0,12:41:11,1,12,41,11
1,11:23:01,2,11,23,1
2,19:51:27,3,19,51,27
3,23:00:36,4,23,0,36
4,18:22:56,5,18,22,56


## 7. Success Check

In [27]:
print("All 6 Dimension Tables Created Successfully!")

All 6 Dimension Tables Created Successfully!


## 3. Create Fact Table

In [28]:
fact = df.copy()

In [29]:
fact = fact.merge(dim_date[['Date_Key','Full_Date']], left_on='Date of Purchase', right_on='Full_Date', how='left')
fact.rename(columns={'Date_Key':'Purchase_Date_Key'}, inplace=True)
fact.drop('Full_Date', axis=1, inplace=True)

fact = fact.merge(dim_date[['Date_Key','Full_Date']], left_on='Date of Journey', right_on='Full_Date', how='left')
fact.rename(columns={'Date_Key':'Journey_Date_Key'}, inplace=True)
fact.drop('Full_Date', axis=1, inplace=True)

In [30]:
fact = fact.merge(dim_station, left_on='Departure Station', right_on='Station_Name', how='left')
fact.rename(columns={'Station_Key':'From_Station_Key'}, inplace=True)
fact.drop('Station_Name', axis=1, inplace=True)

fact = fact.merge(dim_station, left_on='Arrival Destination', right_on='Station_Name', how='left')
fact.rename(columns={'Station_Key':'To_Station_Key'}, inplace=True)
fact.drop('Station_Name', axis=1, inplace=True)

In [31]:
fact = fact.merge(dim_ticket, on=['Ticket Class','Ticket Type','Railcard'], how='left')
fact = fact.merge(dim_payment, on=['Payment Method','Purchase Type'], how='left')
fact = fact.merge(dim_status, on=['Journey Status','Reason for Delay','Refund Request'], how='left')

In [32]:
time_map = dict(zip(dim_time['Time'], dim_time['Time_Key']))

In [33]:
fact['Purchase_Time_Key'] = fact['Time of Purchase'].map(time_map)
fact['Departure_Time_Key'] = fact['Departure Time'].map(time_map)
fact['Arrival_Time_Key'] = fact['Arrival Time'].map(time_map)
fact['Actual_Arrival_Time_Key'] = fact['Actual Arrival Time'].map(time_map)

In [34]:
cols_to_drop = [
    'Date of Purchase',
    'Date of Journey',

    'Time of Purchase',
    'Departure Time',
    'Arrival Time',
    'Actual Arrival Time',

    'Departure Station',
    'Arrival Destination',

    'Ticket Class',
    'Ticket Type',
    'Railcard',

    'Payment Method',
    'Purchase Type',

    'Journey Status',
    'Reason for Delay',
    'Refund Request',
    
    'Arrival_DT',
    'Actual_Arrival_DT'
]

fact_clean = fact.drop(columns=cols_to_drop, errors='ignore')

In [35]:
fact_table = fact_clean[[
    'Transaction ID',

    'Purchase_Date_Key',
    'Journey_Date_Key',

    'Purchase_Time_Key',
    'Departure_Time_Key',
    'Arrival_Time_Key',
    'Actual_Arrival_Time_Key',

    'From_Station_Key',
    'To_Station_Key',

    'Ticket_Key',
    'Payment_Key',
    'Status_Key',

    'Price',
    'Delay Minutes'
]]

## 4. Send Data to Warehouse Database

In [36]:
dw_database = "railway_dw"
server = r".\SQLEXPRESS"

dw_connection_string = (
    f"mssql+pyodbc://{server}/{dw_database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

dw_engine = create_engine(dw_connection_string)

# Send Dimensions
dim_station.to_sql("dim_station", dw_engine, if_exists="replace", index=False)
dim_ticket.to_sql("dim_ticket", dw_engine, if_exists="replace", index=False)
dim_payment.to_sql("dim_payment", dw_engine, if_exists="replace", index=False)
dim_status.to_sql("dim_status", dw_engine, if_exists="replace", index=False)
dim_date.to_sql("dim_date", dw_engine, if_exists="replace", index=False)
dim_time.to_sql("dim_time", dw_engine, if_exists="replace", index=False)

# Send Fact Table
fact_table.to_sql("fact_table", dw_engine, if_exists="replace", index=False)

print("Data Warehouse fully deployed to SQL Server!")

Data Warehouse fully deployed to SQL Server!
